In [5]:
#!/usr/bin/env python3
"""
Build a FAISS index from questions.json (questions-only schema)

Input  (default): questions.json
Output (default): mental_q.index, mental_q_meta.pkl
"""

import json
import pickle
import faiss
import numpy as np
from pathlib import Path
from typing import Dict, Any, List
from sentence_transformers import SentenceTransformer
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ====== CONFIG ======
JSON_FILE     = "questions.json"            # questions-only file
INDEX_FILE    = "mental_q.index"
METADATA_FILE = "mental_q_meta.pkl"
MODEL_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
# ====================

def extract_question_text(item: Dict[str, Any]) -> str:
    """
    Combine relevant fields from a questions-only item into a single string.
    Expected item schema:
      {
        "id": "...",
        "question": { "english": "...", "swahili": "" },
        "category": "anxiety|depression|psychosis|general",
        "tags": ["GAD-7","screening", ...]
      }
    """
    parts: List[str] = []
    q = item.get("question") or {}
    en = (q.get("english") or "").strip()
    sw = (q.get("swahili") or "").strip()
    cat = (item.get("category") or "").strip()
    tags = item.get("tags") or []

    if en: parts.append(en)
    if sw: parts.append(sw)
    if cat: parts.append(f"Category: {cat}")
    if isinstance(tags, list) and tags:
        parts.append("Tags: " + " ".join(t for t in tags if t))

    # Fallback
    if not parts:
        parts.append(item.get("id", ""))

    return " ".join(parts).strip()

def main():
    logger.info("🚀 Building FAISS index from questions.json ...")

    src = Path(JSON_FILE)
    if not src.exists():
        logger.error(f"❌ File not found: {src}")
        return

    data = json.loads(src.read_text(encoding="utf-8"))
    if not isinstance(data, list):
        logger.error("❌ questions.json must be a list")
        return

    # Normalize + collect texts
    items: List[Dict[str, Any]] = []
    texts: List[str] = []
    for i, it in enumerate(data):
        it = dict(it or {})
        if "id" not in it:
            it["id"] = f"q_{i+1}"
        text = extract_question_text(it)
        if text:
            items.append(it)
            texts.append(text)

    logger.info(f"📊 Total questions included: {len(items)}")
    if not items:
        logger.error("❌ No valid questions to index!")
        return

    # Embeddings
    logger.info("🔄 Creating embeddings ...")
    model = SentenceTransformer(MODEL_NAME)
    embeddings = model.encode(texts, show_progress_bar=True)

    # Normalize for cosine similarity
    faiss.normalize_L2(embeddings)
    dim = embeddings.shape[1]

    # FAISS index (Inner Product == cosine after L2 norm)
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype("float32"))
    logger.info(f"✅ FAISS index built with {index.ntotal} questions")

    # Save index + metadata
    faiss.write_index(index, INDEX_FILE)
    logger.info(f"💾 Saved index -> {INDEX_FILE}")

    with open(METADATA_FILE, "wb") as f:
        pickle.dump(
            {"items": items, "embeddings": embeddings, "dimension": dim},
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    logger.info(f"💾 Saved metadata -> {METADATA_FILE}")
    logger.info("🎉 Done!")

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'faiss'